[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/00_intro/00_intro.ipynb)

# 🎬 Módulo 00 · ¿Qué es un sistema de recomendación? El mapa completo

**Curso «Sistemas de Recomendación: de cero a nivel Netflix»** · Bloque I — Fundamentos

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración estimada** | 2–2,5 h (lección) + 2 h (proyecto) |
| **GPU** | No necesaria (CPU). ≈ 0 unidades de Colab |
| **Prerrequisitos** | Python, pandas, NumPy y ML general. **Cero** conocimientos de recomendación |

> Esta es la puerta de entrada del curso. Si solo te quedas con una lección, que sea esta: aquí está el **mapa mental** que vas a ir rellenando durante los 19 módulos siguientes (01–19).
>
> 🧭 Antes de empezar, echa un vistazo a la [guía de estudio](../GUIA_DE_ESTUDIO.md): ruta recomendada, tiempos, mapa de dependencias entre conceptos y glosario ES/EN.

## 🎯 Objetivos de aprendizaje

Al terminar esta lección serás capaz de:

1. **Definir** formalmente el problema de recomendación (predecir una puntuación $s(u, i, c)$ y devolver un top-K) y **distinguirlo** de la clasificación/regresión que ya conoces.
2. **Clasificar** un recomendador como basado en contenido, colaborativo, híbrido o basado en conocimiento, y **calcular a mano** una recomendación de cada uno de los dos primeros tipos.
3. **Diferenciar** feedback explícito e implícito y **enumerar** las señales que registra una plataforma de streaming.
4. **Dibujar** la arquitectura multi-etapa de la industria (*retrieval → ranking → re-ranking*) y **justificar** con números por qué existe.
5. **Describir** el ciclo de vida completo de un recomendador (datos → entrenamiento → evaluación offline → A/B → serving → monitoreo → reentreno) y las capas offline / nearline / online.
6. **Situar** cada herramienta del stack (FAISS, LightGBM, TorchRec, Feast, Kafka, Triton, MLflow…) en su etapa.
7. **Relatar** la historia del campo desde Tapestry (1992) hasta los recomendadores generativos (2024–2026).
8. **Construir** tu primer recomendador (popularidad) sobre MovieLens y **criticarlo**.

In [ ]:
# Colab ya trae numpy, pandas y matplotlib. pyarrow para parquet.
!pip install -q pandas matplotlib pyarrow

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

SEED = 42
rng = np.random.default_rng(SEED)
FAST_DEV_RUN = True          # True: tamaños pequeños para iterar rápido
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})

# Paleta común del curso (una por etapa del embudo)
COL = {"retrieval": "#4C72B0", "ranking": "#DD8452", "rerank": "#55A868",
       "datos": "#8172B3", "gris": "#9E9E9E", "rojo": "#C44E52"}

Vamos a dibujar bastantes diagramas. Para no repetir código, definimos dos ayudantes: una **caja** con texto y una **flecha** entre puntos.

In [ ]:
def caja(ax, x, y, w, h, texto, color="#4C72B0", fs=9, alpha=0.9, tc="white"):
    """Caja redondeada centrada en (x, y)."""
    p = FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                       fc=color, ec="none", alpha=alpha)
    ax.add_patch(p)
    ax.text(x, y, texto, ha="center", va="center", fontsize=fs, color=tc, wrap=True)


def flecha(ax, p1, p2, color="#555555", texto=None, fs=8, rad=0.0, lw=1.5):
    a = FancyArrowPatch(p1, p2, arrowstyle="-|>", mutation_scale=12, color=color, lw=lw,
                        connectionstyle=f"arc3,rad={rad}")
    ax.add_patch(a)
    if texto:
        ax.text((p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2 + 0.12, texto, ha="center",
                fontsize=fs, color=color)


def lienzo(w=10, h=4, xlim=(0, 10), ylim=(0, 4)):
    fig, ax = plt.subplots(figsize=(w, h))
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.axis("off")
    return fig, ax

---
## 1. 💡 Intuición: el problema de la sobreabundancia

Abres Netflix un viernes a las 22:00. El catálogo tiene miles de títulos, pero tú vas a mirar como mucho unas decenas de carátulas antes de decidir… o de cerrar la app. **El recurso escaso no son las películas: es tu atención.**

Un sistema de recomendación (*recommender system*, *recsys*) es el software que decide **qué subconjunto minúsculo del catálogo te enseña, en qué orden y en qué sitio de la pantalla**. Es probablemente el sistema de ML con más impacto económico directo del mundo:

- Netflix: el recomendador *influye* en la elección de **~80 % de las horas reproducidas** (el resto viene de la búsqueda), y estimaban que personalización + recomendaciones les ahorran **más de 1.000 M$ al año** en retención (Gomez-Uribe & Hunt, 2015).
- YouTube: según su CPO Neal Mohan (CES 2018), **más del 70 % del tiempo de visualización** procede de recomendaciones, no de búsquedas.

### La analogía con lo que ya sabes
Piensa en una **matriz enorme** con una fila por usuario y una columna por película. Cada celda dice cuánto le gustó (o si la vio). Casi todas las celdas están **vacías** — nadie ha visto más que una fracción ínfima del catálogo. Recomendar es, en su forma más simple, **rellenar los huecos** y enseñar a cada usuario las celdas vacías con mayor valor predicho.

Veámoslo con un ejemplo de juguete que usaremos toda la lección: 5 usuarios × 6 películas.

In [ ]:
peliculas = ["Toy Story", "Matrix", "Titanic", "Alien", "Notting Hill", "Interstellar"]
usuarios = ["Ana", "Bruno", "Carla", "David", "Elena"]
# Ratings de 1 a 5; NaN = no la ha visto (¡la inmensa mayoría en la vida real!)
R = pd.DataFrame([
    [5, 4, np.nan, np.nan, 1, 5],
    [np.nan, 5, 1, 5, np.nan, 4],
    [4, np.nan, 5, 1, 5, np.nan],
    [np.nan, 4, np.nan, 5, 1, np.nan],
    [5, np.nan, 4, np.nan, 5, 2],
], index=usuarios, columns=peliculas)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
im = ax.imshow(R.values, cmap="YlGnBu", vmin=0.5, vmax=5.5)
for i in range(R.shape[0]):
    for j in range(R.shape[1]):
        v = R.iat[i, j]
        ax.text(j, i, "?" if np.isnan(v) else int(v), ha="center", va="center",
                fontsize=13, color="#C44E52" if np.isnan(v) else "black",
                fontweight="bold" if np.isnan(v) else "normal")
ax.set_xticks(range(6), peliculas, rotation=20); ax.set_yticks(range(5), usuarios)
ax.set_title("Matriz usuario × película: recomendar = predecir los «?» y ordenar")
plt.colorbar(im, ax=ax, label="rating")
plt.tight_layout(); plt.show()
print(f"Celdas observadas: {R.notna().mean().mean():.0%} (en MovieLens-100K ≈ 6 %, en Netflix ≪ 1 %)")

---
> 👀 **Qué debes observar en la matriz:** (1) hay más «?» que números incluso en un juguete de 5×6; en datos reales la proporción observada es del 6 % o menos, así que **el problema es sobre todo de huecos**; (2) los «?» no son ceros: Ana no ha visto *Titanic*, lo que no significa que la odie; (3) para rellenar un «?» puedes mirar **la fila** (qué más le gustó a Ana → contenido) o **la columna y otras filas** (a quién más le gustó *Titanic* y si se parece a Ana → colaborativo). Son las dos familias de la sección 3.

---
## 2. 📐 Teoría formal: ¿qué predice un recomendador?

Sean $\mathcal{U}$ el conjunto de usuarios, $\mathcal{I}$ el catálogo de ítems y $c$ el **contexto** (hora, dispositivo, país, fila de la página…). Un recomendador aprende una **función de puntuación**

$$s_\theta : \mathcal{U} \times \mathcal{I} \times \mathcal{C} \to \mathbb{R}$$

y, para un usuario $u$ en el contexto $c$, devuelve la lista de los $K$ ítems con mayor puntuación entre los **candidatos elegibles** $\mathcal{I}_{u,c} \subseteq \mathcal{I}$ (no vistos, disponibles en su país, aptos para su perfil…):

$$\text{Top-}K(u, c) = \operatorname*{arg\,top\text{-}K}_{i \in \mathcal{I}_{u,c}} \; s_\theta(u, i, c)$$

Dos formulaciones históricas:

| Formulación | Qué predice $s$ | Pérdida típica | Métrica típica | Cuándo |
|---|---|---|---|---|
| **Predicción de rating** (*rating prediction*) | el rating $\hat r_{ui}$ | MSE sobre las celdas observadas | RMSE / MAE | Netflix Prize (2006–09), feedback explícito |
| **Ranking top-N** (*top-N recommendation*) | una puntuación de preferencia cuyo **orden** importa | BPR, softmax, BCE con negativos | Recall@K, NDCG@K | **La industria hoy**, feedback implícito |

### En qué se parece y en qué NO se parece a tu ML de siempre
| ML supervisado clásico | Recomendación |
|---|---|
| Filas i.i.d. con etiqueta | Pares (usuario, ítem) **muy correlacionados**; la "etiqueta" de los no vistos es **desconocida**, no negativa |
| Predices un valor por fila | Lo que importa es el **orden relativo** dentro de la lista de cada usuario |
| Los datos vienen "del mundo" | Los datos los **genera tu propio sistema**: solo ves feedback de lo que recomendaste (*feedback loop*) |
| El test aleatorio es razonable | Hay que respetar el **tiempo**: entrenar con el pasado y evaluar en el futuro (módulo 01) |
| Accuracy / AUC | Métricas **de lista** (NDCG@K) + diversidad, novedad, negocio (módulo 02) |
| Un modelo | Una **cascada** de modelos con presupuestos de latencia distintos (sección 5) |

---
## 3. Tipos de recomendadores

Hay cuatro grandes familias, según **de dónde sale la señal**:

- **Basado en contenido** (*content-based*): "te gustó *Matrix* (ciencia ficción, acción) → te recomiendo otras de ciencia ficción". Solo usa **tu** historial + atributos del ítem. Módulo 03.
- **Filtrado colaborativo** (*collaborative filtering*, CF): "usuarios que se parecen a ti vieron X". No necesita saber nada del ítem: aprende de los **patrones de co-consumo** de millones de personas. Módulos 04, 05 y casi todo el deep learning posterior.
- **Híbrido**: combina ambos (y casi todo lo que hay en producción lo es). Ejemplo: un *two-tower* cuya torre de ítem usa ID + texto + póster (módulos 08 y 11).
- **Basado en conocimiento / reglas** (*knowledge-based*): restricciones explícitas ("casas con 3 habitaciones bajo 300 k€", "solo contenido infantil"). Útil con compras raras (inmuebles, coches) y como capa de **políticas de negocio**.

In [ ]:
fig, ax = lienzo(11, 4.2, (0, 11), (0, 4.2))
caja(ax, 5.5, 3.6, 3.4, 0.6, "Sistemas de recomendación", "#333333", fs=11)
ramas = [
    (1.5, "Basado en contenido\n(atributos del ítem\n+ tu historial)", COL["retrieval"],
     "TF-IDF, BM25, embeddings\nde texto/imagen (CLIP)\n→ Módulo 03"),
    (4.2, "Filtrado colaborativo\n(patrones de\nco-consumo)", COL["ranking"],
     "kNN, EASE, MF/ALS, BPR\ntwo-tower, SASRec, grafos\n→ Módulos 04-05, 08-11"),
    (6.8, "Híbrido\n(contenido + CF\n+ contexto)", COL["rerank"],
     "Wide&Deep, DCN, DIN\nHSTU, foundation models\n→ Módulos 06-07, 11-12"),
    (9.5, "Basado en\nconocimiento / reglas", COL["datos"],
     "Restricciones, filtros,\npolíticas de negocio\n→ Módulos 13-14"),
]
for x, t, c, ej in ramas:
    flecha(ax, (5.5, 3.28), (x, 2.75))
    caja(ax, x, 2.3, 2.3, 0.85, t, c, fs=9)
    ax.text(x, 1.15, ej, ha="center", va="center", fontsize=8.5, color="#333333",
            bbox=dict(boxstyle="round", fc="#F4F4F4", ec="#CCCCCC"))
ax.set_title("Taxonomía de recomendadores (y dónde los verás en el curso)", fontsize=12)
plt.show()

### 🧪 Experimento en papel: recomendar a Ana de dos formas

Ana no ha visto *Titanic* ni *Alien*. ¿Cuál le recomendamos?

**Contenido.** Describimos cada película por sus géneros (vector multi-hot $\mathbf{g}_i$). El perfil de Ana es la media de los géneros de lo que vio, **ponderada** por su rating centrado ($r_{ui} - \bar r_u$): lo que le gustó suma y lo que odió resta. Puntuamos cada película no vista con la similitud coseno:

$$\mathbf{p}_u = \sum_{i \in \mathcal{I}_u} (r_{ui} - \bar r_u)\, \mathbf{g}_i, \qquad s(u,i) = \cos(\mathbf{p}_u, \mathbf{g}_i)$$

**Colaborativo (user-kNN).** Buscamos usuarios parecidos a Ana (coseno entre sus ratings centrados, sobre las películas que ambos vieron) y predecimos su rating como media ponderada de los vecinos:

$$\hat r_{ui} = \bar r_u + \frac{\sum_{v} \text{sim}(u,v)\,(r_{vi} - \bar r_v)}{\sum_{v} |\text{sim}(u,v)|}$$

In [ ]:
generos = pd.DataFrame(  # multi-hot: animación, ciencia ficción, romance, terror, acción
    [[1, 0, 0, 0, 0], [0, 1, 0, 0, 1], [0, 0, 1, 0, 0],
     [0, 1, 0, 1, 1], [0, 0, 1, 0, 0], [0, 1, 0, 0, 0]],
    index=peliculas, columns=["animación", "sci-fi", "romance", "terror", "acción"])

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

u = "Ana"
vistas = R.loc[u].dropna()
no_vistas = R.columns[R.loc[u].isna()]
# --- Contenido ---
perfil = ((vistas - vistas.mean()).values[:, None] * generos.loc[vistas.index].values).sum(0)
s_contenido = {i: cos(perfil, generos.loc[i].values) for i in no_vistas}

# --- Colaborativo (user-kNN) ---
Rc = R.sub(R.mean(axis=1), axis=0)           # centramos por usuario
def sim_usuarios(a, b):
    comunes = R.loc[a].notna() & R.loc[b].notna()
    return cos(Rc.loc[a, comunes].values, Rc.loc[b, comunes].values) if comunes.sum() >= 2 else 0.0
sims = {v: sim_usuarios(u, v) for v in usuarios if v != u}
s_colab = {}
for i in no_vistas:
    vecinos = [v for v in sims if not np.isnan(R.at[v, i])]
    num = sum(sims[v] * Rc.at[v, i] for v in vecinos)
    den = sum(abs(sims[v]) for v in vecinos) + 1e-12
    s_colab[i] = float(np.clip(R.loc[u].mean() + num / den, 1, 5))  # recortamos a la escala 1-5

print("Similitud de Ana con los demás:", {k: round(v, 2) for k, v in sims.items()})
pd.DataFrame({"contenido (coseno)": s_contenido, "colaborativo (rating predicho)": s_colab}).round(2)

Fíjate en lo que ha pasado:
- **Contenido** recomienda *Alien* porque comparte ciencia ficción y acción con *Matrix* e *Interstellar*, que Ana adora. No necesita a nadie más… pero **nunca saldrá de lo que ya conoce** (si solo ves sci-fi, solo te recomendará sci-fi).
- **Colaborativo** llega a la misma conclusión por otro camino: Ana se parece muchísimo a **David** (similitud ≈ 0,98: los dos ponen bien *Matrix* y odian *Notting Hill*) y algo a **Bruno** (≈ 0,38), y a ambos les encantó *Alien*. Fíjate también en **Carla**, de gustos opuestos a Ana (similitud ≈ −0,81): ella odió *Alien*, y con una similitud negativa eso **también suma** a favor de *Alien* (la predicción sale de la escala y la recortamos a 5). En cambio, *Titanic* la adoran Carla y Elena — justo las usuarias con gustos contrarios — y la odió Bruno, así que baja. No ha mirado ni un solo género: solo co-consumo. Su debilidad: un ítem **nuevo**, sin ratings, es invisible (*cold start*).

> ⚠️ Con 5 usuarios todo es frágil: similitudes calculadas sobre 2 películas en común son puro ruido. En el módulo 04 verás el *shrinkage* (penalizar similitudes con poco soporte) y por qué muchos sistemas descartan las similitudes negativas.

Que dos señales independientes coincidan es justo la idea de los **híbridos**. En producción se combinan para cubrir los puntos ciegos de cada una.

---
## 4. Feedback explícito vs implícito

| | Explícito | Implícito |
|---|---|---|
| Ejemplos | estrellas, 👍/👎, "no me interesa" | impresión, clic, play, % visto, tiempo de permanencia (*dwell time*), búsqueda, añadir a "Mi lista", compartir, saltar intro |
| Volumen | escaso (muy pocos usuarios valoran) | **masivo** (cada sesión genera cientos de eventos) |
| Señal negativa | sí (1 estrella) | **ambigua**: no ver ≠ no gustar (quizá nunca se lo enseñaste) |
| Ruido | sesgo de selección (valoras lo que ya elegiste ver) | ruido (autoplay, el niño cogió el mando) |
| Quién lo usa | Netflix Prize, MovieLens | **toda la industria hoy**; Netflix sustituyó las estrellas por 👍/👎 en 2017 |

En CineMatch registraremos una **jerarquía de señales** (un embudo): cada paso es más escaso pero más fiable como señal de preferencia. Elegir **qué evento es el "positivo"** que optimizas es una de las decisiones de producto más importantes (y la verás en los módulos 01, 06 y 07).

In [ ]:
etapas = ["Impresión\n(se mostró la carátula)", "Clic / hover\nen la carátula", "Inicio de\nreproducción",
          "Visto ≥ 70 %", "Valoración\nexplícita (like/dislike)"]
volumen = np.array([1000, 120, 45, 25, 2])   # ILUSTRATIVO: orden de magnitud por cada 1000 impresiones
fig, ax = plt.subplots(figsize=(9, 3.6))
colores = plt.cm.viridis(np.linspace(0.15, 0.85, len(etapas)))
for k, (e, v, c) in enumerate(zip(etapas, volumen, colores)):
    w = np.log10(v) + 0.6
    ax.barh(len(etapas) - k, w, left=-w / 2, color=c, height=0.8)
    ax.text(w / 2 + 0.15, len(etapas) - k, f"{e.replace(chr(10), ' ')}  (~{v})", va="center", fontsize=9)
ax.set_xlim(-2.2, 6); ax.axis("off")
ax.annotate("", xy=(-2, 0.7), xytext=(-2, 5.3), arrowprops=dict(arrowstyle="->", lw=2, color="#555555"))
ax.text(-2.1, 3, "más escasa,\nmás fiable", rotation=90, va="center", ha="right", fontsize=9)
ax.set_title("Embudo de señales de un streaming (cifras ilustrativas por cada 1.000 impresiones, escala log)")
plt.show()

---
> 👀 **Qué debes observar:** cada escalón tiene aproximadamente un orden de magnitud menos eventos que el anterior. Si eliges como positivo la valoración explícita, tendrás ~2 ejemplos por cada 1.000 impresiones; si eliges el clic, 120, pero mucho más ruidosos (*clickbait*). **No hay respuesta gratis**: el módulo 07 entrena varias de estas señales a la vez (multi-tarea) precisamente para no tener que elegir solo una.

---
## 5. 🏭 La arquitectura multi-etapa: el embudo de la industria

Este es **el diagrama más importante del curso**. Imagina que tienes un modelo de ranking buenísimo (un transformer con cientos de *features*). Cuesta, digamos, 20 µs por par (usuario, ítem). Puntuar un catálogo de 10 millones de ítems para **una sola petición** serían 200 s. Tienes ~100–200 ms. **No cabe.**

La solución universal (YouTube, Netflix, Meta, TikTok, Pinterest, Amazon…) es una **cascada**:

1. **Recuperación / generación de candidatos** (*retrieval*, *candidate generation*): modelos **baratos** que, de millones de ítems, sacan **cientos o miles** de candidatos plausibles. Típicamente varias fuentes en paralelo (embeddings + búsqueda ANN, co-visitación, popularidad reciente, "porque viste X"). Optimiza **recall**.
2. **Ranking**: un modelo **caro y preciso** (GBDT, DCN, DIN, multi-tarea) puntúa solo esos candidatos con cientos de *features*. Optimiza la **precisión del orden**.
3. **Re-ranking y políticas**: ajusta la lista final — diversidad, frescura, calibración, reglas de negocio (contenido propio, restricciones legales/parentales), exploración (bandits) — y la compone en la página (filas de la home).

El documento canónico es el paper de YouTube (Covington, Adams & Sargin, RecSys 2016), que describe exactamente dos redes: *candidate generation* y *ranking*.

> 💡 **Ya conoces este patrón si has montado un RAG.** *Retrieval* ≈ el **bi-encoder + base vectorial** que trae los 50 *chunks* más parecidos a la consulta (barato, optimiza recall); *ranking* ≈ el **cross-encoder / reranker** que reordena esos 50 mirando consulta y documento juntos (caro, optimiza precisión); *re-ranking y políticas* ≈ los filtros y la deduplicación que aplicas antes de meter el contexto en el prompt. La diferencia: en recsys la «consulta» es **el usuario y su historia**, no un texto, y el catálogo cambia cada día. Lo verás en detalle en el módulo 08.

In [ ]:
fig, ax = lienzo(12, 5.6, (0, 12), (-1.3, 5))
etapas = [
    ("Catálogo\n10⁶ – 10⁹ ítems", COL["gris"], 4.6, "—"),
    ("① Retrieval\n10³ candidatos", COL["retrieval"], 3.4, "~10-30 ms\nANN (FAISS/HNSW), two-tower,\nco-visitación, popularidad\nMód. 04-05, 08-11"),
    ("② Ranking\n10² ítems", COL["ranking"], 2.3, "~20-80 ms\nGBDT, DCN-v2, DIN,\nmulti-tarea (MMoE)\nMód. 06-07"),
    ("③ Re-ranking\n10-50 ítems", COL["rerank"], 1.4, "~5-10 ms\ndiversidad, calibración,\nreglas, bandits\nMód. 13-14"),
]
for k, (txt, c, h, det) in enumerate(etapas):
    xs = 0.3 + k * 2.55
    poly = plt.Polygon([[xs, 2.5 - h / 2], [xs, 2.5 + h / 2], [xs + 2.2, 2.5 + h / 2 - 0.45],
                        [xs + 2.2, 2.5 - h / 2 + 0.45]], fc=c, alpha=0.9)
    ax.add_patch(poly)
    ax.text(xs + 1.1, 2.5, txt, ha="center", va="center", color="white", fontsize=10, fontweight="bold")
    if det != "—":
        ax.text(xs + 1.1, -0.15, det, ha="center", va="top", fontsize=8, color="#333333")
caja(ax, 11.35, 2.5, 1.1, 1.0, "Página\n(home)", "#333333", fs=9)
flecha(ax, (10.0, 2.5), (10.75, 2.5))
ax.text(6, 4.85, "Cada etapa ve ~10× menos ítems y puede gastar ~10× más cómputo por ítem",
        ha="center", fontsize=11)
ax.text(6, 4.45, "(latencias: órdenes de magnitud típicos, no cifras de ninguna empresa concreta)",
        ha="center", fontsize=8, color="#777777")
plt.show()

### 🧪 Experimento: ¿cuánto ahorra la cascada?

Simulemos un catálogo de ítems con *embeddings* aleatorios y comparemos dos estrategias para servir una petición:
- **Fuerza bruta**: pasar el modelo "caro" (una pequeña red con *features* cruzadas) por **todo** el catálogo.
- **Cascada**: producto escalar barato para quedarnos con 500 candidatos → modelo caro solo sobre ellos → re-ranking con una regla de diversidad (máx. 3 ítems por género).

In [ ]:
N_ITEMS = 100_000 if FAST_DEV_RUN else 1_000_000
D = 32
item_emb = rng.normal(size=(N_ITEMS, D)).astype(np.float32)
item_gen = rng.integers(0, 12, N_ITEMS)                     # "género" de cada ítem
user_emb = rng.normal(size=D).astype(np.float32)
# Modelo "caro": MLP de 3 capas sobre [usuario, ítem, usuario⊙ítem]
W1 = rng.normal(scale=0.1, size=(3 * D, 256)).astype(np.float32)
W2 = rng.normal(scale=0.1, size=(256, 128)).astype(np.float32)
W3 = rng.normal(scale=0.1, size=(128, 1)).astype(np.float32)

def modelo_caro(u, items):
    x = np.hstack([np.repeat(u[None], len(items), 0), items, u[None] * items])
    h = np.maximum(x @ W1, 0); h = np.maximum(h @ W2, 0)
    return (h @ W3).ravel()

def retrieval(u, k=500):
    s = item_emb @ u                                       # barato: un producto escalar por ítem
    return np.argpartition(-s, k)[:k]

def rerank(cands, scores, k=10, max_por_genero=3):
    final, cuenta = [], {}
    for i in cands[np.argsort(-scores)]:
        g = item_gen[i]
        if cuenta.get(g, 0) < max_por_genero:
            final.append(i); cuenta[g] = cuenta.get(g, 0) + 1
        if len(final) == k:
            break
    return final

t = {}
t0 = time.perf_counter(); s_all = modelo_caro(user_emb, item_emb); t["Fuerza bruta\n(modelo caro × catálogo)"] = time.perf_counter() - t0
t0 = time.perf_counter(); cands = retrieval(user_emb); t1 = time.perf_counter()
s_c = modelo_caro(user_emb, item_emb[cands]); t2 = time.perf_counter()
top = [int(i) for i in rerank(cands, s_c)]; t3 = time.perf_counter()
t["① Retrieval"], t["② Ranking (500)"], t["③ Re-ranking"] = t1 - t0, t2 - t1, t3 - t2
for k_, v in t.items():
    print(f"{k_.replace(chr(10), ' '):45s} {v * 1000:8.2f} ms")
print(f"Cascada total: {(t1 - t0 + t2 - t1 + t3 - t2) * 1000:.2f} ms  ·  top-10 final: {top}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
nombres = list(t)
colores = [COL["rojo"], COL["retrieval"], COL["ranking"], COL["rerank"]]
ax.barh(nombres, [v * 1000 for v in t.values()], color=colores)
ax.set_xscale("log"); ax.set_xlabel("milisegundos (escala log)")
ax.invert_yaxis()
ax.set_title(f"Coste de servir 1 petición sobre {N_ITEMS:,} ítems: fuerza bruta vs cascada")
for y, v in enumerate(t.values()):
    ax.text(v * 1000 * 1.1, y, f"{v * 1000:.2f} ms", va="center", fontsize=9)
plt.tight_layout(); plt.show()

Con un modelo de ranking realista (cientos de *features* que hay que **buscar** en un *feature store*, no solo multiplicar) la diferencia es aún mayor. Y la cascada tiene un precio: **lo que el retrieval no trae, el ranking no lo puede arreglar**. Por eso las métricas de cada etapa son distintas: *recall@1000* en retrieval, NDCG en ranking.

> 🏭 **Variante 2D: la home de Netflix.** No es una lista, es una **cuadrícula**: filas temáticas ("Porque viste…", "Tendencias", "Continuar viendo") y dentro de cada fila un orden. Hay que elegir **qué filas**, **en qué orden** y **qué ítems** en cada una, sin repetir. Es optimización a nivel de página (módulo 13).

---
## 6. El ciclo de vida completo

Un recomendador no es un modelo: es un **bucle**. Lo que recomiendas hoy cambia lo que los usuarios ven, eso cambia los logs, y con esos logs entrenas el modelo de mañana. Si vienes de MLOps, esto te sonará — pero con una diferencia crucial: aquí **el modelo genera sus propios datos de entrenamiento** (*feedback loop*).

In [ ]:
pasos = [("Logs de eventos\n(Kafka)", COL["datos"]), ("Datos y features\n(feature store)", COL["datos"]),
         ("Entrenamiento\n(GPU, TorchRec)", COL["retrieval"]), ("Evaluación\noffline", COL["retrieval"]),
         ("Test A/B\n(online)", COL["ranking"]), ("Serving\n(retrieval→ranking)", COL["ranking"]),
         ("Monitoreo\n(drift, negocio)", COL["rerank"]), ("Reentreno\n(programado/trigger)", COL["rerank"])]
mods = ["01", "01·16", "03-12", "02", "15", "16", "17", "17"]
fig, ax = lienzo(8, 7.2, (-4, 4), (-3.8, 3.6))
n = len(pasos); R0 = 2.7
ang = np.pi / 2 - np.arange(n) * 2 * np.pi / n
for k in range(n):
    x, y = R0 * np.cos(ang[k]), R0 * np.sin(ang[k])
    caja(ax, x, y, 1.9, 0.8, pasos[k][0], pasos[k][1], fs=8.5)
    ax.text(x, y - 0.55, f"mód. {mods[k]}", ha="center", fontsize=7.5, color="#555555")
    a1, a2 = ang[k] - 0.33, ang[(k + 1) % n] + 0.33
    flecha(ax, (R0 * np.cos(a1), R0 * np.sin(a1)), (R0 * np.cos(a2), R0 * np.sin(a2)), rad=-0.25)
ax.text(0, 0.25, "USUARIOS", ha="center", fontsize=13, fontweight="bold")
ax.text(0, -0.35, "lo que recomiendas\nhoy son los datos\nde mañana", ha="center", fontsize=9, color=COL["rojo"])
ax.set_title("Ciclo de vida de un recomendador en producción", fontsize=12)
plt.show()

### Offline, nearline y online (la arquitectura de Netflix)
Netflix popularizó (Amatriain & Basilico, *Netflix Tech Blog*, 2013) la división del cómputo en tres capas según su **frescura** y su **presupuesto de latencia**:

- **Offline** (batch, horas): entrenar modelos, precalcular *embeddings* y listas de candidatos, generar *features* agregadas. Sin límite de latencia, todo el histórico.
- **Nearline** (segundos–minutos): reaccionar a eventos recientes (acabas de terminar una serie → recalcular "porque viste…") sin bloquear la petición. Streaming con Kafka/Flink.
- **Online** (milisegundos): lo que ocurre durante la petición: retrieval ANN, ranking con *features* en tiempo real, re-ranking. Debe tener **fallbacks** (si el ranker cae, sirves listas precalculadas).

In [ ]:
fig, ax = lienzo(11, 4.4, (0, 11), (0, 4.4))
capas = [(3.6, "OFFLINE  (horas)", "Entrenar modelos · precalcular embeddings,\ncandidatos y features agregadas · backfills",
          COL["retrieval"], "Spark · Airflow/Metaflow · GPUs\nMLflow · data warehouse"),
         (2.2, "NEARLINE  (segundos-minutos)", "Reaccionar a eventos recientes · actualizar\nfeatures y listas sin bloquear la petición",
          COL["ranking"], "Kafka · Flink · Redis\nfeature store (Feast)"),
         (0.8, "ONLINE  (milisegundos)", "Retrieval ANN · ranking con features en\ntiempo real · re-ranking · fallbacks",
          COL["rerank"], "FAISS/HNSW · Triton · FastAPI\nRedis · caches")]
for y, titulo, desc, c, herr in capas:
    caja(ax, 1.9, y, 3.3, 1.05, titulo, c, fs=10)
    ax.text(3.9, y, desc, va="center", fontsize=9)
    ax.text(8.4, y, herr, va="center", fontsize=8.5, color="#444444",
            bbox=dict(boxstyle="round", fc="#F4F4F4", ec="#CCCCCC"))
ax.annotate("", xy=(0.12, 0.35), xytext=(0.12, 4.1), arrowprops=dict(arrowstyle="->", lw=2, color="#555"))
ax.text(0.0, 2.2, "más fresco · menos cómputo", rotation=90, va="center", ha="right", fontsize=8)
ax.set_title("Las tres capas de cómputo (Amatriain & Basilico, Netflix 2013)", fontsize=12)
plt.show()

---
## 7. 🧰 El stack de herramientas, etapa por etapa

No hace falta que memorices esto ahora: es el índice de lo que vas a tocar con las manos durante el curso.

| Etapa | Qué resuelve | Herramientas de industria (y open source) | Módulo |
|---|---|---|---|
| Logging y datos | eventos de usuario, joins, splits | Kafka / Redpanda, Spark, pandas/Polars, Parquet, Pandera / Great Expectations | 01, 16, 17 |
| Features | *features* consistentes en entrenamiento y serving | **Feast**, Redis, Tecton | 16 |
| Retrieval | millones → miles en ms | EASE/ALS (`implicit`), **two-tower** (PyTorch/TorchRec), **FAISS**, ScaNN, hnswlib, Milvus/Qdrant/pgvector/Vespa | 04, 05, 08 |
| Ranking | orden preciso de cientos de candidatos | **LightGBM**/XGBoost (lambdarank), DCN-v2, DIN, MMoE/PLE (PyTorch, TorchRec, NVIDIA Merlin) | 06, 07 |
| Secuencial / generativo | historia del usuario como secuencia | SASRec, BERT4Rec, HSTU, semantic IDs (RQ-VAE) | 09, 11 |
| Re-ranking | diversidad, calibración, exploración | MMR, DPP, bandits (Vowpal Wabbit, Open Bandit Pipeline) | 13, 14 |
| LLMs y agentes | cold start, explicaciones, conversación | sentence-transformers, LLMs, **LangGraph** | 03, 12 |
| Evaluación | offline rigurosa | **ranx**, RecBole, Elliot, Microsoft Recommenders, recpack | 02 |
| Experimentación | A/B, interleaving | plataformas internas; statsmodels/scipy | 15 |
| Serving | latencia y escala | FastAPI, **NVIDIA Triton**, BentoML, Ray Serve, ONNX | 16 |
| MLOps | tracking, orquestación, monitoreo | **MLflow**, Airflow/Prefect/**Metaflow** (creado en Netflix), **Evidently**, Prometheus+Grafana | 17 |

---
## 8. 📜 Treinta y cinco años en una línea temporal

- **1992 — Tapestry** (Xerox PARC, Goldberg et al.): acuñan *collaborative filtering*; los usuarios anotaban correos y filtraban por las anotaciones de otros.
- **1994 — GroupLens** (Resnick et al.): CF automático para noticias de Usenet. El grupo de Minnesota publicará después **MovieLens** (1997–hoy).
- **2003 — Amazon item-to-item CF** (Linden, Smith & York): en vez de buscar usuarios parecidos, precalcula ítems parecidos → escala a decenas de millones de clientes.
- **2006–2009 — Netflix Prize**: 1 M$ a quien mejorase un 10 % el RMSE de Cinematch. Lo ganó *BellKor's Pragmatic Chaos* (2009). Legado: factorización matricial, *ensembles*… y la lección de que el RMSE no era lo que importaba (módulo 02).
- **2008 — ALS para feedback implícito** (Hu, Koren & Volinsky). **2009 — BPR** (Rendle et al.): el giro hacia el ranking con señales implícitas.
- **2010 — Factorization Machines** (Rendle). **2014 — GBDT+LR** en Facebook Ads (He et al.).
- **2016 — YouTube DNN** (Covington et al.): la arquitectura de dos etapas con redes profundas. **Wide & Deep** (Google Play, Cheng et al.).
- **2018 — PinSage** (Pinterest, grafos a escala web), **SASRec** (Kang & McAuley, self-attention secuencial), **DIN** (Alibaba).
- **2019 — Two-tower con corrección de sesgo de muestreo** (Yi et al., YouTube) y la **crisis de reproducibilidad** (Ferrari Dacrema et al.): muchos modelos neuronales no superaban a baselines bien ajustados.
- **2023 — TIGER / semantic IDs** (Rajput et al., Google): recuperación **generativa**.
- **2024 — HSTU / Generative Recommenders** (Zhai et al., Meta): transformers de billones de parámetros, *scaling laws* en recomendación.
- **2025 — Foundation model de recomendación de Netflix** (Netflix Tech Blog) y **OneRec** (Kuaishou): un modelo grande y unificado sustituye a muchos modelos pequeños.

In [ ]:
hitos = [(1992, "Tapestry\n(CF)", 0), (1994, "GroupLens", 1), (1997, "MovieLens", 2),
         (2003, "Amazon\nitem-to-item", 0), (2006, "Netflix Prize\n(arranca)", 1), (2008, "ALS implícito", 2),
         (2009, "BPR · fin del\nNetflix Prize", 0), (2010, "Factorization\nMachines", 1),
         (2016, "YouTube DNN ·\nWide&Deep", 2), (2018, "PinSage ·\nSASRec · DIN", 0),
         (2019, "Two-tower logQ ·\ncrisis reproducib.", 1), (2023, "TIGER\n(semantic IDs)", 2),
         (2024, "HSTU\n(Meta)", 0), (2025, "Foundation model\nNetflix · OneRec", 1)]
eras = [(1990, 2005, "Memoria y vecinos", COL["datos"]), (2005, 2015, "Factorización latente", COL["retrieval"]),
        (2015, 2022, "Deep learning", COL["ranking"]), (2022, 2027, "Transformers\ny generativos", COL["rerank"])]
fig, ax = plt.subplots(figsize=(13, 4.6))
for a, b, nombre, c in eras:
    ax.add_patch(plt.Rectangle((a, -0.18), b - a, 0.36, color=c, alpha=0.9))
    ax.text((a + b) / 2, 0, nombre, ha="center", va="center", color="white", fontsize=8.5, fontweight="bold")
for k, (año, txt, nivel) in enumerate(hitos):
    signo = 1 if k % 2 == 0 else -1
    y = signo * [0.6, 1.1, 1.6][nivel]
    ax.plot([año, año], [0.18 * signo, y - 0.2 * signo], color="#777777", lw=0.8)
    ax.plot(año, 0.18 * signo, "o", color="#555555", ms=3)
    ax.text(año, y, f"{año}\n{txt}", ha="center", va="center", fontsize=7.5)
ax.set_xlim(1989, 2027.5); ax.set_ylim(-2.0, 2.0); ax.axis("off")
ax.set_title("Historia de los sistemas de recomendación", fontsize=12)
plt.tight_layout(); plt.show()

---
## 9. 🧪 Primer contacto con datos reales: MovieLens

**MovieLens** (GroupLens, Universidad de Minnesota; Harper & Konstan, 2015) es el "MNIST" de la recomendación: ratings de 1 a 5 estrellas de usuarios reales de [movielens.org](https://movielens.org). Hay varias versiones en <https://files.grouplens.org/datasets/movielens/>: **100K** (943 usuarios, 1.682 películas), **1M**, **latest-small**, **25M** y **32M**. Será el dataset de CineMatch durante todo el curso.

La celda siguiente escribe en tu sesión el módulo `cinematch_data.py`, el pipeline de datos del curso (lo construirás tú en el módulo 01). Si no hay red, `load_movielens` genera **datos sintéticos** con el mismo esquema para que el notebook nunca se bloquee.

In [ ]:
#@title 🔧 Utilidades del curso: cinematch_data.py { display-mode: "form" }
# Escribe los módulos reutilizables del curso en el directorio actual.
# Última versión en GitHub (alternativa):
#   !wget -q https://raw.githubusercontent.com/padillacm/IA-practice/master/recsys-course/01_data/cinematch_data.py
_SOURCES = {}
_SOURCES['cinematch_data.py'] = '"""cinematch_data — pipeline de datos de CineMatch (módulo 01 del curso).\n\nInterfaz estable que reutilizan los módulos posteriores:\n\n    ratings, items = load_movielens("100k")      # o "1m", "latest-small", "25m", "32m"\n    users = load_users("100k")                    # demografía (solo 100k y 1m; si no, None)\n    inter = to_implicit(ratings, threshold=4.0)   # feedback implícito (rating >= 4)\n    inter = filter_k_core(inter, min_user=5, min_item=5)\n    train, val, test = temporal_split(inter, val_frac=0.1, test_frac=0.1)\n    train, test = leave_one_out_split(inter)      # último ítem de cada usuario a test\n    train, test = random_split(inter, test_frac=0.2)\n    X, u2i, i2i = to_csr(train)                   # matriz usuario×ítem scipy.sparse\n    recs = recommend_popular(train, users=test.user_id.unique(), k=10)\n    # recs: dict[user_id -> list[item_id]] ordenado de más a menos recomendado\n\nConvenciones de columnas: user_id (int), item_id (int), rating (float),\ntimestamp (int, segundos Unix). items: item_id, title, year, genres (str "A|B").\n\nSi GroupLens no responde, 100k y 1m se descargan de un espejo público en GitHub\n(mismo contenido); si tampoco hay red, `load_movielens` genera un dataset\nsintético con el mismo esquema (`make_synthetic_movielens`) y avisa.\n"""\nfrom __future__ import annotations\n\nimport io\nimport urllib.request\nimport warnings\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\n\nBASE_URL = "https://files.grouplens.org/datasets/movielens"\nSIZES = {\n    "100k": "ml-100k",\n    "1m": "ml-1m",\n    "latest-small": "ml-latest-small",\n    "25m": "ml-25m",\n    "32m": "ml-32m",\n}\nGENRES_100K = [\n    "unknown", "Action", "Adventure", "Animation", "Children\'s", "Comedy", "Crime",\n    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",\n    "Romance", "Sci-Fi", "Thriller", "War", "Western",\n]\n\n\n# ----------------------------------------------------------------------------\n# Descarga y carga\n# ----------------------------------------------------------------------------\ndef download_movielens(size: str = "100k", data_dir: str | Path = "data") -> Path:\n    """Descarga y descomprime MovieLens desde GroupLens. Devuelve la carpeta."""\n    if size not in SIZES:\n        raise ValueError(f"size debe ser uno de {list(SIZES)}")\n    name = SIZES[size]\n    data_dir = Path(data_dir)\n    folder = data_dir / name\n    if folder.exists():\n        return folder\n    data_dir.mkdir(parents=True, exist_ok=True)\n    url = f"{BASE_URL}/{name}.zip"\n    print(f"Descargando {url} ...")\n    try:\n        with urllib.request.urlopen(url, timeout=60) as resp:\n            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(data_dir)\n    except Exception as e:\n        if size not in MIRRORS:\n            raise\n        print(f"GroupLens no responde ({e.__class__.__name__}); probando espejo público en GitHub…")\n        _download_mirror(size, folder)\n    return folder\n\n\n# Espejos públicos (mismo contenido que GroupLens) por si files.grouplens.org no responde.\nMIRRORS = {\n    "100k": "https://raw.githubusercontent.com/RUCAIBox/RecBole/master/dataset/ml-100k",\n    "1m": "https://raw.githubusercontent.com/khanhnamle1994/movielens/master",\n}\n\n\ndef _write_dat(df: pd.DataFrame, path: Path) -> None:\n    """Escribe en el formato \'::\' de MovieLens-1M."""\n    lines = ("::".join(map(str, row)) for row in df.itertuples(index=False))\n    path.write_text("\\n".join(lines) + "\\n", encoding="latin-1", errors="replace")\n\n\ndef _download_mirror(size: str, folder: Path) -> None:\n    """Descarga el espejo y lo reescribe con los nombres y el formato originales de\n    GroupLens, para que `_read_folder` y `load_users` funcionen sin cambios."""\n    import csv\n    base, tmp = MIRRORS[size], folder.with_name(folder.name + ".tmp")\n    tmp.mkdir(parents=True, exist_ok=True)\n    if size == "100k":\n        rd = lambda f: pd.read_csv(f"{base}/ml-100k.{f}", sep="\\t", quoting=csv.QUOTE_NONE)\n        inter, it, us = rd("inter"), rd("item"), rd("user")\n        inter.columns, it.columns = ["u", "i", "r", "t"], ["item_id", "title", "year", "genres"]\n        inter.astype("int64").to_csv(tmp / "u.data", sep="\\t", header=False, index=False)\n        year = pd.to_numeric(it["year"], errors="coerce")\n        title = [f"{t} ({int(y)})" if y == y else t for t, y in zip(it["title"], year)]\n        flags = pd.DataFrame({g: it["genres"].str.split(" ").apply(lambda gs: int(g in gs))\n                              for g in GENRES_100K})\n        u_item = pd.concat([it[["item_id"]], pd.Series(title, name="title"),\n                            pd.DataFrame({"rd": "", "vrd": "", "url": ""}, index=it.index), flags], axis=1)\n        u_item.to_csv(tmp / "u.item", sep="|", header=False, index=False, encoding="latin-1",\n                      errors="replace", quoting=csv.QUOTE_NONE, escapechar="\\\\")\n        us.to_csv(tmp / "u.user", sep="|", header=False, index=False)\n    else:  # 1m\n        rd = lambda f: pd.read_csv(f"{base}/{f}.csv", sep="\\t", index_col=0, encoding="latin-1")\n        _write_dat(rd("ratings")[["user_id", "movie_id", "rating", "timestamp"]], tmp / "ratings.dat")\n        _write_dat(rd("movies")[["movie_id", "title", "genres"]], tmp / "movies.dat")\n        _write_dat(rd("users")[["user_id", "gender", "age", "occupation", "zipcode"]], tmp / "users.dat")\n    tmp.rename(folder)\n\n\ndef _split_title_year(titles: pd.Series) -> tuple[pd.Series, pd.Series]:\n    year = titles.str.extract(r"\\((\\d{4})\\)\\s*$")[0].astype("float")\n    clean = titles.str.replace(r"\\s*\\(\\d{4}\\)\\s*$", "", regex=True)\n    return clean, year\n\n\ndef _read_folder(size: str, folder: Path) -> tuple[pd.DataFrame, pd.DataFrame]:\n    if size == "100k":\n        ratings = pd.read_csv(folder / "u.data", sep="\\t",\n                              names=["user_id", "item_id", "rating", "timestamp"])\n        cols = ["item_id", "title", "release_date", "video_release_date", "url"] + GENRES_100K\n        raw = pd.read_csv(folder / "u.item", sep="|", names=cols, encoding="latin-1")\n        flags = raw[GENRES_100K].values.astype(bool)\n        genres = ["|".join(g for g, f in zip(GENRES_100K, row) if f) or "unknown" for row in flags]\n        items = pd.DataFrame({"item_id": raw["item_id"], "title": raw["title"].fillna(""),\n                              "genres": genres})\n    elif size == "1m":\n        ratings = pd.read_csv(folder / "ratings.dat", sep="::", engine="python",\n                              names=["user_id", "item_id", "rating", "timestamp"])\n        items = pd.read_csv(folder / "movies.dat", sep="::", engine="python",\n                            names=["item_id", "title", "genres"], encoding="latin-1")\n    else:\n        ratings = pd.read_csv(folder / "ratings.csv").rename(\n            columns={"userId": "user_id", "movieId": "item_id"})\n        items = pd.read_csv(folder / "movies.csv").rename(columns={"movieId": "item_id"})\n    items["title"], items["year"] = _split_title_year(items["title"].astype(str))\n    return ratings, items[["item_id", "title", "year", "genres"]]\n\n\ndef load_movielens(size: str = "100k", data_dir: str | Path = "data",\n                   synthetic_fallback: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Devuelve (ratings, items) con el esquema estándar de CineMatch.\n\n    ratings: user_id, item_id, rating (float32), timestamp (int64), ordenado por tiempo.\n    items:   item_id, title, year, genres ("Action|Comedy").\n    """\n    try:\n        folder = download_movielens(size, data_dir)\n        ratings, items = _read_folder(size, folder)\n    except Exception as e:  # sin red, URL caída, etc.\n        if not synthetic_fallback:\n            raise\n        warnings.warn(f"No se pudo descargar MovieLens-{size} ({e!r}). "\n                      "Usando datos SINTÉTICOS con el mismo esquema.")\n        ratings, items = make_synthetic_movielens()\n    ratings = ratings.astype({"user_id": "int64", "item_id": "int64",\n                              "rating": "float32", "timestamp": "int64"})\n    ratings = ratings.sort_values(["timestamp", "user_id", "item_id"], kind="stable")\n    return ratings.reset_index(drop=True), items.reset_index(drop=True)\n\n\ndef load_users(size: str = "100k", data_dir: str | Path = "data") -> pd.DataFrame | None:\n    """Demografía de usuarios (user_id, age, gender, occupation). Solo 100k y 1m."""\n    try:\n        folder = download_movielens(size, data_dir)\n    except Exception:\n        return None\n    if size == "100k":\n        return pd.read_csv(folder / "u.user", sep="|",\n                           names=["user_id", "age", "gender", "occupation", "zip"])\n    if size == "1m":\n        return pd.read_csv(folder / "users.dat", sep="::", engine="python",\n                           names=["user_id", "gender", "age", "occupation", "zip"])\n    return None\n\n\ndef make_synthetic_movielens(n_users: int = 943, n_items: int = 1682,\n                             n_ratings: int = 100_000, seed: int = 42\n                             ) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Dataset sintético con forma de MovieLens: long tail, actividad sesgada,\n    gustos latentes por género y deriva temporal (los ítems nuevos van llegando)."""\n    rng = np.random.default_rng(seed)\n    genres = GENRES_100K[1:]\n    t0, t1 = 874_724_710, 893_286_638  # mismo rango temporal que ML-100K\n    # Ítems: género principal, popularidad intrínseca (Zipf) y fecha de llegada\n    item_genre = rng.integers(0, len(genres), n_items)\n    item_pop = 1.0 / np.arange(1, n_items + 1) ** 1.1\n    rng.shuffle(item_pop)\n    item_arrival = t0 + (t1 - t0) * np.where(rng.random(n_items) < 0.5, 0.0,\n                                             rng.uniform(0, 0.95, n_items))  # catálogo inicial + estrenos\n    # Usuarios: actividad log-normal, afinidad por géneros y fecha de alta\n    activity = rng.lognormal(0, 1.0, n_users)\n    activity = np.maximum(20, activity / activity.sum() * n_ratings).astype(int)\n    taste = rng.dirichlet(np.full(len(genres), 0.3), n_users)\n    user_start = t0 + (t1 - t0) * rng.uniform(0, 0.9, n_users)\n    item_quality = rng.normal(0, 0.45, n_items)   # unas películas son mejores que otras\n    user_bias = rng.normal(0, 0.35, n_users)      # hay usuarios generosos y exigentes\n    rows = []\n    for u in range(n_users):\n        start = user_start[u]\n        end = min(t1, start + rng.uniform(1, 120) * 86_400)\n        avail = item_arrival <= end\n        n = int(min(activity[u], avail.sum() // 2))\n        ts = np.sort(rng.uniform(start, end, n))\n        age_days = np.maximum(0, end - item_arrival) / 86_400\n        hype = np.where(item_arrival > t0, 1 + 30 * np.exp(-age_days / 30), 1.0)  # los estrenos tienen su momento\n        w = item_pop * hype * (0.15 + taste[u][item_genre]) * avail\n        chosen = rng.choice(n_items, size=n, replace=False, p=w / w.sum())\n        ts = np.maximum(ts, item_arrival[chosen] + 3600)\n        aff = taste[u][item_genre[chosen]] * len(genres)\n        r = np.clip(np.round(3.2 + 0.6 * np.log1p(aff) + item_quality[chosen] + user_bias[u]\n                             + rng.normal(0, 0.7, n)), 1, 5)\n        rows.append(pd.DataFrame({"user_id": u + 1, "item_id": chosen + 1,\n                                  "rating": r, "timestamp": ts.astype(np.int64)}))\n    ratings = pd.concat(rows, ignore_index=True)\n    second = rng.integers(0, len(genres), n_items)\n    items = pd.DataFrame({\n        "item_id": np.arange(1, n_items + 1),\n        "title": [f"Película sintética {i}" for i in range(1, n_items + 1)],\n        "year": (1930 + 66 * rng.beta(4, 1.5, n_items)).round(),\n        "genres": [genres[a] if a == b else f"{genres[a]}|{genres[b]}"\n                   for a, b in zip(item_genre, second)],\n    })\n    return ratings, items\n\n\n# ----------------------------------------------------------------------------\n# Limpieza y señales\n# ----------------------------------------------------------------------------\ndef to_implicit(ratings: pd.DataFrame, threshold: float = 4.0) -> pd.DataFrame:\n    """Convierte ratings explícitos en interacciones implícitas positivas\n    (rating >= threshold). Mantiene la columna rating por si se quiere ponderar."""\n    out = ratings[ratings["rating"] >= threshold].copy()\n    return out.reset_index(drop=True)\n\n\ndef dedup_interactions(df: pd.DataFrame, keep: str = "last") -> pd.DataFrame:\n    """Elimina pares (usuario, ítem) duplicados quedándose con el primero/último en el tiempo."""\n    return (df.sort_values("timestamp", kind="stable")\n              .drop_duplicates(["user_id", "item_id"], keep=keep)\n              .reset_index(drop=True))\n\n\ndef filter_k_core(df: pd.DataFrame, min_user: int = 5, min_item: int = 5,\n                  max_iter: int = 100) -> pd.DataFrame:\n    """k-core iterativo: repite hasta que todo usuario tenga >= min_user\n    interacciones y todo ítem >= min_item (filtrar uno puede romper el otro)."""\n    for _ in range(max_iter):\n        n0 = len(df)\n        ic = df["item_id"].map(df["item_id"].value_counts())\n        df = df[ic >= min_item]\n        uc = df["user_id"].map(df["user_id"].value_counts())\n        df = df[uc >= min_user]\n        if len(df) == n0:\n            break\n    return df.reset_index(drop=True)\n\n\n# ----------------------------------------------------------------------------\n# Splits\n# ----------------------------------------------------------------------------\ndef _drop_cold(train: pd.DataFrame, other: pd.DataFrame, users: bool, items: bool) -> pd.DataFrame:\n    if users:\n        other = other[other["user_id"].isin(train["user_id"].unique())]\n    if items:\n        other = other[other["item_id"].isin(train["item_id"].unique())]\n    return other.reset_index(drop=True)\n\n\ndef random_split(df: pd.DataFrame, test_frac: float = 0.2, seed: int = 42,\n                 drop_cold: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Split aleatorio de interacciones (¡filtra el futuro al train! solo como contraejemplo)."""\n    mask = np.random.default_rng(seed).random(len(df)) < test_frac\n    train, test = df[~mask].reset_index(drop=True), df[mask]\n    return train, _drop_cold(train, test, drop_cold, drop_cold)\n\n\ndef leave_one_out_split(df: pd.DataFrame, n_test: int = 1, min_train: int = 1\n                        ) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Las últimas n_test interacciones (en el tiempo) de cada usuario van a test.\n    Ojo: no hay un corte temporal global → el train de un usuario puede contener\n    interacciones posteriores al test de otro (leakage temporal entre usuarios)."""\n    df = df.sort_values(["user_id", "timestamp"], kind="stable")\n    rank_from_end = df.groupby("user_id").cumcount(ascending=False)\n    n_user = df.groupby("user_id")["item_id"].transform("size")\n    is_test = (rank_from_end < n_test) & (n_user >= n_test + min_train)\n    return df[~is_test].reset_index(drop=True), df[is_test].reset_index(drop=True)\n\n\ndef temporal_split(df: pd.DataFrame, val_frac: float = 0.1, test_frac: float = 0.1,\n                   drop_cold_users: bool = True, drop_cold_items: bool = True\n                   ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """Split temporal GLOBAL: train = pasado, val = siguiente val_frac del tiempo\n    (por cuantiles de eventos), test = último test_frac. Es lo más parecido a\n    producción: entrenas con todo lo anterior a T y sirves después de T.\n    Devuelve (train, val, test); val puede estar vacío si val_frac == 0."""\n    ts = df["timestamp"]\n    t_test = ts.quantile(1 - test_frac)\n    t_val = ts.quantile(1 - test_frac - val_frac) if val_frac > 0 else t_test\n    train = df[ts < t_val].reset_index(drop=True)\n    val = df[(ts >= t_val) & (ts < t_test)]\n    test = df[ts >= t_test]\n    val = _drop_cold(train, val, drop_cold_users, drop_cold_items)\n    # Para test, "conocido" = visto en train o val (en producción reentrenas antes de servir)\n    seen = pd.concat([train, val])\n    test = _drop_cold(seen, test, drop_cold_users, drop_cold_items)\n    return train, val, test\n\n\n# ----------------------------------------------------------------------------\n# Matrices dispersas\n# ----------------------------------------------------------------------------\ndef build_mappings(df: pd.DataFrame) -> tuple[dict, dict]:\n    """Mapeos id original -> índice contiguo 0..n-1 (usuarios e ítems)."""\n    users = np.sort(df["user_id"].unique())\n    items = np.sort(df["item_id"].unique())\n    return ({u: i for i, u in enumerate(users)}, {it: j for j, it in enumerate(items)})\n\n\ndef to_csr(df: pd.DataFrame, user2idx: dict | None = None, item2idx: dict | None = None,\n           value_col: str | None = None) -> tuple[sp.csr_matrix, dict, dict]:\n    """Matriz usuario×ítem CSR. value_col=None → 1.0 por interacción (implícito).\n    Las filas/ítems fuera del mapeo se descartan. Devuelve (X, user2idx, item2idx)."""\n    if user2idx is None or item2idx is None:\n        user2idx, item2idx = build_mappings(df)\n    r = df["user_id"].map(user2idx)\n    c = df["item_id"].map(item2idx)\n    ok = r.notna() & c.notna()\n    vals = np.ones(ok.sum(), dtype=np.float32) if value_col is None \\\n        else df.loc[ok, value_col].to_numpy(np.float32)\n    X = sp.csr_matrix((vals, (r[ok].astype(int), c[ok].astype(int))),\n                      shape=(len(user2idx), len(item2idx)))\n    X.sum_duplicates()\n    return X, user2idx, item2idx\n\n\n# ----------------------------------------------------------------------------\n# Baselines\n# ----------------------------------------------------------------------------\ndef _seen_by_user(train: pd.DataFrame) -> dict:\n    return train.groupby("user_id")["item_id"].agg(set).to_dict()\n\n\ndef _topk_excluding(ranked: np.ndarray, seen: set, k: int) -> list:\n    out = []\n    for it in ranked:\n        if it not in seen:\n            out.append(int(it))\n            if len(out) == k:\n                break\n    return out\n\n\ndef popularity_ranking(train: pd.DataFrame, window_days: float | None = None,\n                       half_life_days: float | None = None) -> np.ndarray:\n    """Ítems ordenados por popularidad. window_days: solo cuenta los últimos N días.\n    half_life_days: pondera cada evento por 0.5**(edad/half_life) (recencia suave)."""\n    df = train\n    t_max = df["timestamp"].max()\n    if window_days is not None:\n        df = df[df["timestamp"] >= t_max - window_days * 86_400]\n    if half_life_days is not None:\n        age = (t_max - df["timestamp"]) / 86_400\n        w = 0.5 ** (age / half_life_days)\n        scores = w.groupby(df["item_id"]).sum()\n    else:\n        scores = df["item_id"].value_counts()\n    return scores.sort_values(ascending=False, kind="stable").index.to_numpy()\n\n\ndef recommend_popular(train: pd.DataFrame, users, k: int = 10, exclude_seen: bool = True,\n                      window_days: float | None = None,\n                      half_life_days: float | None = None) -> dict:\n    """Top-K más populares (opcionalmente recientes) para cada usuario de `users`."""\n    ranked = popularity_ranking(train, window_days, half_life_days)\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(ranked, seen.get(u, set()), k) for u in users}\n\n\ndef recommend_popular_by_segment(train: pd.DataFrame, users, segment_of: dict, k: int = 10,\n                                 exclude_seen: bool = True, min_segment_events: int = 50) -> dict:\n    """Popularidad dentro del segmento del usuario (p. ej. edad, país, género favorito).\n    `segment_of`: dict user_id -> segmento. Segmentos con pocos eventos → popularidad global."""\n    seg = train["user_id"].map(segment_of)\n    global_rank = popularity_ranking(train)\n    ranks = {}\n    for s, g in train.groupby(seg):\n        if len(g) >= min_segment_events:\n            r = g["item_id"].value_counts().index.to_numpy()\n            # completar con el ranking global para no quedarnos cortos\n            ranks[s] = np.concatenate([r, global_rank[~np.isin(global_rank, r)]])\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(ranks.get(segment_of.get(u), global_rank), seen.get(u, set()), k)\n            for u in users}\n\n\ndef recommend_random(train: pd.DataFrame, users, k: int = 10, seed: int = 42,\n                     exclude_seen: bool = True) -> dict:\n    """Recomendador aleatorio: el suelo absoluto de cualquier comparación."""\n    rng = np.random.default_rng(seed)\n    items = train["item_id"].unique()\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(rng.permutation(items), seen.get(u, set()), k) for u in users}\n\n\n# ----------------------------------------------------------------------------\n# Persistencia\n# ----------------------------------------------------------------------------\ndef save_splits(out_dir: str | Path, **frames: pd.DataFrame) -> Path:\n    """Guarda cada DataFrame como parquet: save_splits("data/cinematch", train=..., test=...)."""\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    for name, f in frames.items():\n        f.to_parquet(out_dir / f"{name}.parquet", index=False)\n    return out_dir\n\n\ndef load_splits(out_dir: str | Path, names=("train", "val", "test", "items")) -> dict:\n    out_dir = Path(out_dir)\n    return {n: pd.read_parquet(out_dir / f"{n}.parquet") for n in names\n            if (out_dir / f"{n}.parquet").exists()}\n\n\n# ----------------------------------------------------------------------------\n# Pipeline completo de CineMatch (una llamada)\n# ----------------------------------------------------------------------------\ndef prepare_cinematch(size: str = "100k", threshold: float | None = 4.0, min_user: int = 5,\n                      min_item: int = 5, val_frac: float = 0.1, test_frac: float = 0.1,\n                      data_dir: str | Path = "data", out_dir: str | Path | None = None) -> dict:\n    """Pipeline estándar del curso:\n    carga → dedup → implícito (rating >= threshold; None = no filtrar) → split temporal\n    global → k-core SOLO sobre train (no miramos el futuro para filtrar) → val/test\n    restringidos a usuarios e ítems conocidos en train.\n\n    Devuelve {"train", "val", "test", "items", "ratings"} (DataFrames) y, si\n    out_dir no es None, los guarda en parquet."""\n    ratings, items = load_movielens(size, data_dir)\n    inter = dedup_interactions(ratings)\n    if threshold is not None:\n        inter = to_implicit(inter, threshold)\n    train, val, test = temporal_split(inter, val_frac, test_frac,\n                                      drop_cold_users=False, drop_cold_items=False)\n    train = filter_k_core(train, min_user, min_item)\n    val = _drop_cold(train, val, True, True)\n    test = _drop_cold(train, test, True, True)\n    out = {"train": train, "val": val, "test": test, "items": items, "ratings": ratings}\n    if out_dir is not None:\n        save_splits(out_dir, train=train, val=val, test=test, items=items)\n    return out\n'
for _name, _src in _SOURCES.items():
    with open(_name, 'w', encoding='utf-8') as _fh:
        _fh.write(_src)
    print('✔ escrito', _name)

In [ ]:
from cinematch_data import load_movielens

ratings, items = load_movielens("100k")
n_u, n_i = ratings.user_id.nunique(), ratings.item_id.nunique()
print(f"{len(ratings):,} ratings · {n_u:,} usuarios · {n_i:,} películas")
print(f"Densidad de la matriz: {len(ratings) / (n_u * n_i):.2%}  →  dispersión {1 - len(ratings) / (n_u * n_i):.2%}")
print(f"Periodo: {pd.to_datetime(ratings.timestamp.min(), unit='s'):%Y-%m-%d} → "
      f"{pd.to_datetime(ratings.timestamp.max(), unit='s'):%Y-%m-%d}")
ratings.merge(items, on="item_id").head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
ratings.rating.value_counts().sort_index().plot.bar(ax=axes[0], color=COL["retrieval"])
axes[0].set(title="Distribución de ratings", xlabel="estrellas", ylabel="nº de ratings")
axes[0].tick_params(axis="x", rotation=0)

pop = ratings.item_id.value_counts().values
axes[1].plot(np.arange(1, len(pop) + 1), pop, color=COL["ranking"])
corte = int(0.2 * len(pop))
axes[1].fill_between(np.arange(1, corte + 1), pop[:corte], color=COL["ranking"], alpha=0.3)
axes[1].set(title=f"Long tail: el 20 % más popular acapara el {pop[:corte].sum() / pop.sum():.0%} de ratings",
            xlabel="película (ordenada por popularidad)", ylabel="nº de ratings")

act = ratings.user_id.value_counts().values
axes[2].hist(act, bins=50, color=COL["rerank"])
axes[2].set(title="Actividad por usuario", xlabel="nº de ratings del usuario", ylabel="nº de usuarios", yscale="log")
plt.tight_layout(); plt.show()

Tres hechos que verás en **todos** los datasets de recomendación:
1. **Ratings sesgados al alto**: la gente valora lo que eligió ver, y elige lo que cree que le gustará (sesgo de selección, módulo 01).
2. **Long tail**: unas pocas películas concentran la mayoría de las interacciones; miles tienen un puñado.
3. **Actividad muy desigual**: unos pocos usuarios aportan muchísimos datos; la mayoría, pocos.

### Tu primer recomendador: popularidad
El recomendador más simple que existe: a todo el mundo, las películas más vistas que aún no ha visto. Parece una broma, pero es **el baseline que hay que batir siempre**, y a menudo cuesta.

In [ ]:
def recomendar_populares(ratings: pd.DataFrame, user_id: int, k: int = 10) -> pd.DataFrame:
    vistas = set(ratings.loc[ratings.user_id == user_id, "item_id"])
    ranking = ratings.item_id.value_counts()
    top = [i for i in ranking.index if i not in vistas][:k]
    return items.set_index("item_id").loc[top, ["title", "year", "genres"]].assign(
        n_ratings=ranking.loc[top].values)

recomendar_populares(ratings, user_id=1)

¿Qué le pasa a este recomendador?
- **No es personal**: cualquier usuario recibe casi la misma lista (solo cambia lo que ya ha visto).
- **Cobertura ridícula**: de 1.682 películas solo muestra unas pocas decenas en todo el sistema → la cola larga nunca se descubre.
- **Se autorrefuerza** (*rich get richer*): lo popular se muestra más → se ve más → es más popular.
- **Ignora el tiempo**: "lo más visto desde 1997" no es "lo que se está viendo esta semana".

…y aun así, en el proyecto de esta sesión verás que bate a una alternativa aparentemente más "inteligente". En el módulo 02 aprenderás a medir todo esto con rigor.

---
## 10. 🏭 En producción

- **Netflix**: la home es una cuadrícula de filas personalizadas; cada fila la genera un algoritmo distinto (PVR — *personalized video ranker*, *Top-N*, *trending*, *continue watching*, *because you watched*) y otra capa elige y ordena las filas (Gomez-Uribe & Hunt, 2015). Incluso la **carátula** que ves se personaliza con bandits (Netflix Tech Blog, *Artwork Personalization at Netflix*, 2017; módulo 14). En 2025 describieron un **foundation model** de recomendación entrenado sobre el historial de interacciones a escala de LLM que alimenta a muchos modelos aguas abajo.
- **YouTube**: arquitectura de dos etapas con redes profundas (Covington et al., 2016); el objetivo del ranker es el **tiempo de visualización esperado**, no el clic. Después: two-tower con corrección logQ (Yi et al., 2019), multi-tarea MMoE (Zhao et al., 2019).
- **Amazon**: item-to-item CF precalculado offline (Linden et al., 2003) — "los clientes que compraron esto también compraron…". Sigue siendo un generador de candidatos muy potente.
- **Pinterest**: PinSage, GCN sobre un grafo de miles de millones de nodos (Ying et al., 2018).
- **TikTok (ByteDance)**: *Monolith* (Liu et al., 2022), entrenamiento en tiempo real con tablas de *embeddings* sin colisiones: el modelo se actualiza en minutos con lo que acabas de ver.
- **Meta**: HSTU (Zhai et al., 2024) — un transformer generativo para ranking y retrieval que sustituye *features* hechas a mano por la secuencia cruda de acciones.

---
## 11. 🧠 Secretos de la élite

1. **La popularidad es un baseline brutalmente fuerte.** En muchos datasets públicos, un recomendador de "lo más popular (reciente)" bate a modelos personalizados mal ajustados. Si tu modelo nuevo no gana con claridad a popularidad en un split temporal, no tienes modelo. Ferrari Dacrema et al. (2019) intentaron reproducir 18 modelos neuronales de conferencias top: solo 7 eran reproducibles y **6 de esos 7 perdían frente a baselines simples bien ajustados** (kNN, grafos, popularidad en algunos casos).
2. **La métrica offline no es el negocio.** Los equipos de élite usan la evaluación offline para **filtrar** ideas y el test A/B como **juez final**: mejoras offline que no se trasladan online (*offline-online gap*) son lo normal, no la excepción. Netflix lo explica en Gomez-Uribe & Hunt (2015).
3. **El objetivo importa más que la arquitectura.** YouTube anunció en 2012 que pasaba a optimizar **tiempo de visualización** en lugar de clics, para premiar vídeos que la gente realmente ve y no *clickbait*. Elegir el "positivo" (clic, play, 70 % visto, retención a 30 días) cambia el producto más que cambiar de modelo.
4. **El retrieval pone el techo.** Los equipos dedican tanto esfuerzo a la generación de candidatos (varias fuentes, *recall@1000*) como al ranker, porque un ítem que no entra en el embudo no existe.
5. **Tu sistema contamina sus propios datos.** Solo observas feedback de lo que mostraste. Entrenar ingenuamente sobre esos logs amplifica sesgos y homogeneiza el consumo (Chaney, Stewart & Engelhardt, RecSys 2018). Por eso los grandes reservan tráfico de **exploración** o inserciones aleatorias (KuaiRand, 2022, se construyó así).
6. **La frescura es una *feature*, no un detalle.** Un modelo entrenado con logs históricos aprende la popularidad *media* del pasado y infravalora lo recién llegado. YouTube lo resolvió con una sola *feature*, la **edad del ejemplo de entrenamiento** (*example age*), que se pone a 0 en serving: el modelo aprende la dependencia temporal de la popularidad y deja de penalizar el contenido nuevo (Covington et al., 2016, sección 3.3). TikTok lleva la idea al extremo actualizando el modelo en minutos (Monolith, Liu et al., 2022).
7. **La página es el producto, no la lista.** Optimizar cada fila por separado produce una home repetitiva; los sistemas maduros optimizan la página entera (diversidad entre filas, deduplicación, posición).

## ⚠️ Errores comunes

- Tratar los ítems no vistos como **negativos** ("no la vio → no le gusta"). Lo más probable es que **nunca se la enseñaste**.
- Evaluar con un **split aleatorio** y creerte el número: mezclas futuro y pasado (*leakage*, módulo 01).
- Optimizar RMSE cuando tu producto muestra una **lista top-K** (módulo 02).
- Empezar por un modelo profundo sin haber medido antes **popularidad, popularidad reciente e item-kNN**.
- Olvidar las **reglas de negocio** (contenido no disponible en el país, restricciones parentales, ítems ya vistos) y descubrirlas en producción.
- Diseñar solo el modelo y no el **bucle**: logging de impresiones, monitoreo y reentreno (módulos 16–17).

---
## 📝 Autoevaluación

**1.** ¿Por qué no se puede usar directamente el mejor modelo de ranking sobre todo el catálogo?
<details><summary>Respuesta</summary>Por latencia y coste: puntuar millones de ítems con un modelo caro por cada petición no cabe en ~100 ms. Se usa una cascada: retrieval barato (miles de candidatos) → ranking caro (cientos) → re-ranking (decenas).</details>

**2.** Un usuario no ha visto *Alien*. ¿Es un ejemplo negativo?
<details><summary>Respuesta</summary>No necesariamente. En feedback implícito, "no visto" mezcla "no le interesa" con "nunca se le mostró" o "no lo conoce". Por eso se habla de datos <i>missing not at random</i> y se usan negativos muestreados o ponderados (módulos 01 y 05).</details>

**3.** ¿Qué ventaja tiene el filtrado colaborativo sobre el basado en contenido, y qué desventaja?
<details><summary>Respuesta</summary>Ventaja: descubre relaciones que los atributos no capturan (gustos "raros" compartidos) y produce serendipia. Desventaja: no puede recomendar ítems nuevos sin interacciones (cold start de ítem) ni servir bien a usuarios nuevos.</details>

**4.** Asigna cada herramienta a su etapa: FAISS, LightGBM-lambdarank, MMR, Feast, Kafka.
<details><summary>Respuesta</summary>FAISS → retrieval (búsqueda ANN). LightGBM-lambdarank → ranking. MMR → re-ranking (diversidad). Feast → features (offline/online). Kafka → logging/streaming de eventos (nearline).</details>

**5.** ¿Qué diferencia hay entre las capas offline, nearline y online?
<details><summary>Respuesta</summary>Frescura y presupuesto de latencia: offline = batch de horas con todo el histórico (entrenar, precalcular); nearline = reaccionar en segundos/minutos a eventos sin bloquear la petición; online = milisegundos durante la petición, con fallbacks.</details>

**6.** ¿Qué lección dejó el Netflix Prize además de la factorización matricial?
<details><summary>Respuesta</summary>Que optimizar RMSE sobre ratings no es lo mismo que mejorar el producto: Netflix no llevó a producción el ensemble ganador completo porque la ganancia no compensaba el coste de ingeniería, y su negocio había migrado al streaming con señales implícitas y listas top-K.</details>

**7.** ¿Por qué es peligroso entrenar cada día con los logs del recomendador actual sin más?
<details><summary>Respuesta</summary>Porque solo contienen feedback de lo que el sistema eligió mostrar (sesgo de exposición). El modelo nuevo aprende a imitar al viejo y los sesgos se amplifican (feedback loop, homogeneización). Se mitiga con exploración, corrección por propensión (IPS) y datos aleatorizados.</details>

**8. (Razonamiento)** CineMatch tiene 50.000 títulos y un ranker que tarda 50 µs por par. Quieres responder en 100 ms y dedicar como mucho 40 ms al ranking. ¿Cuántos candidatos debe traer, como máximo, el *retrieval*? ¿Qué le pasa al sistema si el retrieval no incluye la película que el usuario habría visto?
<details><summary>Respuesta</summary>40 ms / 50 µs = 800 candidatos (en la práctica menos, por el coste de buscar <i>features</i>). Puntuar los 50.000 costaría 2,5 s. Si el retrieval no trae la película correcta, <b>ningún ranker puede recuperarla</b>: el recall del retrieval es el techo de todo el embudo (por eso se mide <i>recall@1000</i> en esa etapa).</details>

**9. (Transferencia)** Tu equipo propone evaluar el primer modelo personalizado con un split aleatorio 80/20 de las valoraciones, «como en cualquier clasificador». Da dos razones de la tabla de la sección 2 por las que eso puede engañar.
<details><summary>Respuesta</summary>(1) Las interacciones no son i.i.d.: un split aleatorio mete en train valoraciones <b>posteriores</b> a las de test (el modelo «ve el futuro», p. ej. la popularidad final de una película). (2) Lo que importa es el orden de la lista de cada usuario en el futuro, no el error por fila; un split temporal simula mejor lo que pasará al desplegar. Lo medirás con números en los módulos 01 y 02.</details>

---
## 📚 Referencias

**Papers**
- Goldberg, Nichols, Oki & Terry (1992). *Using collaborative filtering to weave an information tapestry*. Communications of the ACM 35(12). <https://doi.org/10.1145/138859.138867>
- Resnick, Iacovou, Suchak, Bergstrom & Riedl (1994). *GroupLens: an open architecture for collaborative filtering of netnews*. CSCW. <https://doi.org/10.1145/192844.192905>
- Linden, Smith & York (2003). *Amazon.com recommendations: item-to-item collaborative filtering*. IEEE Internet Computing 7(1). <https://doi.org/10.1109/MIC.2003.1167344>
- Hu, Koren & Volinsky (2008). *Collaborative Filtering for Implicit Feedback Datasets*. ICDM. <https://doi.org/10.1109/ICDM.2008.22>
- Rendle, Freudenthaler, Gantner & Schmidt-Thieme (2009). *BPR: Bayesian Personalized Ranking from Implicit Feedback*. UAI. <https://arxiv.org/abs/1205.2618>
- Harper & Konstan (2015). *The MovieLens Datasets: History and Context*. ACM TiiS 5(4). <https://doi.org/10.1145/2827872>
- Gomez-Uribe & Hunt (2015). *The Netflix Recommender System: Algorithms, Business Value, and Innovation*. ACM TMIS 6(4). <https://doi.org/10.1145/2843948>
- Covington, Adams & Sargin (2016). *Deep Neural Networks for YouTube Recommendations*. RecSys. <https://doi.org/10.1145/2959100.2959190>
- Chaney, Stewart & Engelhardt (2018). *How Algorithmic Confounding in Recommendation Systems Increases Homogeneity and Decreases Utility*. RecSys. <https://arxiv.org/abs/1710.11214>
- Ferrari Dacrema, Cremonesi & Jannach (2019). *Are We Really Making Much Progress? A Worrying Analysis of Recent Neural Recommendation Approaches*. RecSys. <https://arxiv.org/abs/1907.06902>
- Yi et al. (2019). *Sampling-Bias-Corrected Neural Modeling for Large Corpus Item Recommendations*. RecSys. <https://doi.org/10.1145/3298689.3346996>
- Liu et al. (2022). *Monolith: Real Time Recommendation System With Collisionless Embedding Table*. <https://arxiv.org/abs/2209.07663>
- Rajput et al. (2023). *Recommender Systems with Generative Retrieval* (TIGER). NeurIPS. <https://arxiv.org/abs/2305.05065>
- Zhai et al. (2024). *Actions Speak Louder than Words: Trillion-Parameter Sequential Transducers for Generative Recommendations* (HSTU). ICML. <https://arxiv.org/abs/2402.17152>

**Blogs de ingeniería**
- Amatriain & Basilico (2012). *Netflix Recommendations: Beyond the 5 stars*. Netflix Tech Blog. <https://netflixtechblog.com/netflix-recommendations-beyond-the-5-stars-part-1-55838468f429>
- Amatriain & Basilico (2013). *System Architectures for Personalization and Recommendation*. Netflix Tech Blog. <https://netflixtechblog.com/system-architectures-for-personalization-and-recommendation-e081aa94b5d8>
- Chandrashekar, Amat, Basilico & Jebara (2017). *Artwork Personalization at Netflix*. Netflix Tech Blog. <https://netflixtechblog.com/artwork-personalization-c589f074ad76>
- Netflix (2025). *Foundation Model for Personalized Recommendation*. Netflix Tech Blog (<https://netflixtechblog.com>).
- YouTube Creator Blog (2012). *YouTube Now: Why We Focus on Watch Time*.

**Libros y cursos**
- Aggarwal (2016). *Recommender Systems: The Textbook*. Springer.
- Ricci, Rokach & Shapira (eds.) (2022). *Recommender Systems Handbook*, 3.ª ed. Springer.
- Falk (2019). *Practical Recommender Systems*. Manning.